# CTIP 2-Class CLIP Notebook

This notebook combines the **training pipeline** and the **realtime camera system** for the 2-class AI/CV prototype:

- `TouchingPlants`
- `TouchingWildlife`

It is adapted from the uploaded Python scripts and reorganized with headings and comments for easier review, testing, and future modification.

To download all the files:
https://drive.google.com/drive/folders/1CQjiJNnVJYRK3W0qHI2qcUDwjG2cA0qV?usp=sharing

## Part A. Training the 2-Class CLIP Model

This part:
1. Renames the dataset images
2. Splits the dataset into train and test sets
3. Builds and trains the CLIP-based classifier
4. Evaluates the trained model
5. Saves the trained weights


### Imports / Setup

This section is taken from the training script and kept with comments for clarity.

In [1]:
import random
import shutil
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from transformers import AutoProcessor, CLIPVisionModelWithProjection

### PATHS

This section is taken from the training script and kept with comments for clarity.

In [2]:
PROJECT_DIR = Path(r"C:\COS30049 Assignment")
DATASET_DIR = PROJECT_DIR / "Datasets"
SPLIT_DIR = PROJECT_DIR / "Datasets_split_2class"
ARTIFACT_DIR = PROJECT_DIR / "Artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = ARTIFACT_DIR / "clip_2class_touching_species.pt"

### CONFIG

This section is taken from the training script and kept with comments for clarity.

In [3]:
MODEL_NAME = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_FOLDERS = {
    "TouchingPlants": "touching_plants",
    "TouchingWildlife": "touching_wildlife",
}

CLASS_NAMES = [
    "TouchingPlants",
    "TouchingWildlife",
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

TRAIN_RATIO = 0.90
TEST_RATIO = 0.10
RANDOM_SEED = 42

BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
NUM_WORKERS = 0

### STEP 1: RENAME IMAGES

This section is taken from the training script and kept with comments for clarity.

In [4]:
def rename_images():
    print("=== STEP 1: Renaming images ===")
    for folder_name, prefix in CLASS_FOLDERS.items():
        folder = DATASET_DIR / folder_name
        if not folder.exists():
            print(f"Missing folder: {folder}")
            continue

        files = sorted([
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXT
        ])

        for idx, f in enumerate(files, start=1):
            new_name = f"{prefix}_{idx:04d}{f.suffix.lower()}"
            new_path = folder / new_name
            if f != new_path:
                f.rename(new_path)

        print(f"Renamed {len(files)} images in {folder_name}")

### STEP 2: SHOW COUNTS

This section is taken from the training script and kept with comments for clarity.

In [5]:
def show_counts():
    print("\n=== STEP 2: Dataset counts ===")
    for folder_name in CLASS_NAMES:
        folder = DATASET_DIR / folder_name
        count = len([
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXT
        ])
        print(f"{folder_name}: {count}")

### STEP 3: SPLIT DATASET

This section is taken from the training script and kept with comments for clarity.

In [6]:
def split_dataset():
    print("\n=== STEP 3: Splitting dataset (90/10) ===")
    random.seed(RANDOM_SEED)

    for split in ["train", "test"]:
        for cls in CLASS_NAMES:
            (SPLIT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

    for cls in CLASS_NAMES:
        src_dir = DATASET_DIR / cls
        files = [
            f for f in src_dir.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXT
        ]
        random.shuffle(files)

        n = len(files)
        n_train = int(n * TRAIN_RATIO)
        train_files = files[:n_train]
        test_files = files[n_train:]

        for f in train_files:
            shutil.copy2(f, SPLIT_DIR / "train" / cls / f.name)

        for f in test_files:
            shutil.copy2(f, SPLIT_DIR / "test" / cls / f.name)

        print(f"{cls}: train={len(train_files)}, test={len(test_files)}")

### DATASET CLASS

This section is taken from the training script and kept with comments for clarity.

In [7]:
class ImageFolderDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.root_dir = Path(root_dir)
        self.processor = processor
        self.samples = []

        for class_name in CLASS_NAMES:
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue

            for img_path in class_dir.iterdir():
                if img_path.is_file() and img_path.suffix.lower() in VALID_EXT:
                    self.samples.append((str(img_path), CLASS_TO_IDX[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        return pixel_values, label

### MODEL

This section is taken from the training script and kept with comments for clarity.

In [8]:
class CLIPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.clip = CLIPVisionModelWithProjection.from_pretrained(
            model_name,
            use_safetensors=True
        )

        for p in self.clip.parameters():
            p.requires_grad = False

        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, pixel_values):
        outputs = self.clip(pixel_values=pixel_values)
        image_embeds = outputs.image_embeds
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        logits = self.classifier(image_embeds)
        return logits

### TRAIN / TEST HELPERS

This section is taken from the training script and kept with comments for clarity.

In [9]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for pixel_values, labels in tqdm(loader, leave=False):
        pixel_values = pixel_values.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(training):
            logits = model(pixel_values)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)

        total_loss += loss.item() * labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, macro_f1, all_labels, all_preds

### MAIN

This section is taken from the training script and kept with comments for clarity.

In [10]:
def main():
    rename_images()
    show_counts()
    split_dataset()

    print("\n=== STEP 4: Loading processor and datasets ===")
    processor = AutoProcessor.from_pretrained(MODEL_NAME)

    train_ds = ImageFolderDataset(SPLIT_DIR / "train", processor)
    test_ds = ImageFolderDataset(SPLIT_DIR / "test", processor)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f"Train samples: {len(train_ds)}")
    print(f"Test samples : {len(test_ds)}")

    print("\n=== STEP 5: Building model ===")
    model = CLIPClassifier(MODEL_NAME, len(CLASS_NAMES)).to(DEVICE)

    # weighted loss in case class counts are still imbalanced
    train_labels = [label for _, label in train_ds.samples]
    label_counts = Counter(train_labels)
    print("Train label counts:")
    for idx, count in sorted(label_counts.items()):
        print(f"  {CLASS_NAMES[idx]}: {count}")

    total = len(train_labels)
    num_classes = len(CLASS_NAMES)
    class_weights = []
    for i in range(num_classes):
        count = label_counts[i]
        weight = total / (num_classes * count)
        class_weights.append(weight)

    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    print("Class weights:", class_weights)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)

    print("\n=== STEP 6: Training ===")
    for epoch in range(EPOCHS):
        train_loss, train_acc, train_f1, _, _ = run_epoch(
            model, train_loader, criterion, optimizer=optimizer
        )

        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"  Train Loss    : {train_loss:.4f}")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Train Macro F1: {train_f1:.4f}")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"\nSaved model to: {MODEL_SAVE_PATH}")

    print("\n=== STEP 7: Final test evaluation ===")
    model.eval()
    test_loss, test_acc, test_f1, test_labels, test_preds = run_epoch(
        model, test_loader, criterion, optimizer=None
    )

    print("\n=== TEST RESULTS ===")
    print(f"Test Loss    : {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(test_labels, test_preds))


if __name__ == "__main__":
    main()

=== STEP 1: Renaming images ===
Renamed 232 images in TouchingPlants
Renamed 172 images in TouchingWildlife

=== STEP 2: Dataset counts ===
TouchingPlants: 232
TouchingWildlife: 172

=== STEP 3: Splitting dataset (90/10) ===
TouchingPlants: train=208, test=24
TouchingWildlife: train=154, test=18

=== STEP 4: Loading processor and datasets ===


Train samples: 362
Test samples : 42

=== STEP 5: Building model ===


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.position_ids                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_no

Train label counts:
  TouchingPlants: 208
  TouchingWildlife: 154
Class weights: tensor([0.8702, 1.1753])

=== STEP 6: Training ===


...

Epoch 17/20
  Train Loss    : 0.1559
  Train Accuracy: 0.9862
  Train Macro F1: 0.9858


Epoch 18/20
  Train Loss    : 0.1485
  Train Accuracy: 0.9862
  Train Macro F1: 0.9858


Epoch 19/20
  Train Loss    : 0.1403
  Train Accuracy: 0.9862
  Train Macro F1: 0.9858


Epoch 20/20
  Train Loss    : 0.1346
  Train Accuracy: 0.9862
  Train Macro F1: 0.9858

Saved model to: C:\COS30049 Assignment\Artifacts\clip_2class_touching_species.pt

=== STEP 7: Final test evaluation ===



=== TEST RESULTS ===
Test Loss    : 0.1608
Test Accuracy: 0.9762
Test Macro F1: 0.9755

Classification Report:
                  precision    recall  f1-score   support

  TouchingPlants       0.96      1.00      0.98        24
TouchingWildlife       1.00      0.94      0.97        18

        accuracy                           0.98        42
       macro avg       0.98      0.97      0.98        42
    weighted avg       0.98      0.98      0.98        42


Confusion Matrix:
[[24  0]
 [ 1 17]]


## Part B. Realtime Camera System

This part:
1. Loads the trained 2-class model
2. Detects hands with MediaPipe
3. Crops the detected hand region
4. Classifies the crop as `TouchingPlants` or `TouchingWildlife`
5. Applies rejection logic and rolling alert logic
6. Saves evidence images and JSON metadata


### Imports / Setup

This section is taken from the realtime camera script and kept with comments for clarity.

In [1]:
import cv2
import json
import time
from pathlib import Path
from collections import deque, Counter

import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoProcessor, CLIPVisionModelWithProjection

import mediapipe as mp
from mediapipe.tasks.python import vision

### PATHS

This section is taken from the realtime camera script and kept with comments for clarity.

In [2]:
PROJECT_DIR = Path(r"C:\COS30049 Assignment")
MODEL_PATH = PROJECT_DIR / "Artifacts" / "clip_2class_touching_species.pt"
HAND_MODEL_PATH = PROJECT_DIR / "Models" / "hand_landmarker.task"
ALERT_DIR = PROJECT_DIR / "Alerts"
ALERT_DIR.mkdir(parents=True, exist_ok=True)

### CONFIG

This section is taken from the realtime camera script and kept with comments for clarity.

In [3]:
MODEL_NAME = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FRAME_SKIP = 3
ROLLING_WINDOW = 5
ALERT_MIN_TOUCHING = 3
COOLDOWN_SECONDS = 5

HAND_BOX_PADDING = 30
MIN_HAND_BOX_SIZE = 100

# Rejection / unknown logic
CLASS_CONF_THRESHOLD = 0.75
MIN_MARGIN = 0.15

# Box sanity filtering
MIN_BOX_AREA_RATIO = 0.01
MAX_BOX_AREA_RATIO = 0.35

CLASS_NAMES = [
    "TouchingPlants",
    "TouchingWildlife",
]

FRIENDLY_LABELS = {
    "TouchingPlants": "Touching Plants",
    "TouchingWildlife": "Touching Wildlife",
    "Unknown": "Unknown / No Alert",
}

### MODEL

This section is taken from the realtime camera script and kept with comments for clarity.

In [4]:
class CLIPClassifier(nn.Module):
    def __init__(self, model_name: str, num_classes: int):
        super().__init__()
        self.clip = CLIPVisionModelWithProjection.from_pretrained(
            model_name,
            use_safetensors=True
        )

        for p in self.clip.parameters():
            p.requires_grad = False

        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, pixel_values):
        outputs = self.clip(pixel_values=pixel_values)
        image_embeds = outputs.image_embeds
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        logits = self.classifier(image_embeds)
        return logits

### SAFETY CHECKS

This section is taken from the realtime camera script and kept with comments for clarity.

In [5]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not HAND_MODEL_PATH.exists():
    raise FileNotFoundError(f"Hand landmarker model not found: {HAND_MODEL_PATH}")

### LOAD MODEL

This section is taken from the realtime camera script and kept with comments for clarity.

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = CLIPClassifier(MODEL_NAME, len(CLASS_NAMES)).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("===================================================")
print("Realtime 2-class monitor starting...")
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Model path:", MODEL_PATH)
print("Hand model:", HAND_MODEL_PATH)
print("Alert dir:", ALERT_DIR)
print("Press 'q' to quit | Press 's' to save snapshot")
print("===================================================")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Realtime 2-class monitor starting...
CUDA available: False
Model device: cpu
Model path: C:\COS30049 Assignment\Artifacts\clip_2class_touching_species.pt
Hand model: C:\COS30049 Assignment\Models\hand_landmarker.task
Alert dir: C:\COS30049 Assignment\Alerts
Press 'q' to quit | Press 's' to save snapshot


### MEDIAPIPE HAND LANDMARKER

This section is taken from the realtime camera script and kept with comments for clarity.

In [7]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(HAND_MODEL_PATH)),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

hand_landmarker = HandLandmarker.create_from_options(hand_options)

### HELPERS

This section is taken from the realtime camera script and kept with comments for clarity.

In [8]:
def draw_fullscreen_border(frame, color=(0, 0, 255), thickness=18):
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0, 0), (w - 1, h - 1), color, thickness)


def get_bbox_from_landmarks(landmarks, frame_w, frame_h, padding=30):
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]

    x1 = max(0, int(min(xs) * frame_w) - padding)
    y1 = max(0, int(min(ys) * frame_h) - padding)
    x2 = min(frame_w - 1, int(max(xs) * frame_w) + padding)
    y2 = min(frame_h - 1, int(max(ys) * frame_h) + padding)

    if (x2 - x1) < MIN_HAND_BOX_SIZE or (y2 - y1) < MIN_HAND_BOX_SIZE:
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        half = MIN_HAND_BOX_SIZE // 2
        x1 = max(0, cx - half)
        y1 = max(0, cy - half)
        x2 = min(frame_w - 1, cx + half)
        y2 = min(frame_h - 1, cy + half)

    return x1, y1, x2, y2


def get_box_color(pred_class: str):
    if pred_class == "TouchingPlants":
        return (0, 165, 255)   # orange
    if pred_class == "TouchingWildlife":
        return (0, 0, 255)     # red
    return (180, 180, 180)     # gray


def classify_crop(crop_bgr):
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(rgb)

    pixel_values = processor(images=image, return_tensors="pt")["pixel_values"].to(DEVICE)

    with torch.no_grad():
        logits = model(pixel_values)
        probs = torch.softmax(logits, dim=1)[0].cpu().tolist()
        pred_idx = int(torch.argmax(logits, dim=1).item())

    probs_dict = {name: float(p) for name, p in zip(CLASS_NAMES, probs)}
    pred_class = CLASS_NAMES[pred_idx]

    sorted_probs = sorted(probs, reverse=True)
    top1 = float(sorted_probs[0])
    top2 = float(sorted_probs[1]) if len(sorted_probs) > 1 else 0.0
    margin = top1 - top2

    return pred_class, probs_dict, top1, margin


def is_valid_touch_detection(det, frame_w, frame_h):
    x1, y1, x2, y2 = det["bbox"]
    area_ratio = ((x2 - x1) * (y2 - y1)) / float(frame_w * frame_h)

    return (
        det["top1"] >= CLASS_CONF_THRESHOLD
        and det["margin"] >= MIN_MARGIN
        and MIN_BOX_AREA_RATIO <= area_ratio <= MAX_BOX_AREA_RATIO
    )


def save_alert_frame(frame, detections, history, touch_votes, event_class, prefix="alert"):
    timestamp = time.strftime("%Y-%m-%d_%H-%M-%S")
    image_path = ALERT_DIR / f"{timestamp}_{prefix}_{event_class}.jpg"
    json_path = ALERT_DIR / f"{timestamp}_{prefix}_{event_class}.json"

    ok = cv2.imwrite(str(image_path), frame)
    print(f"[SAVE] Image path: {image_path}")
    print(f"[SAVE] Image save success: {ok}")

    payload = {
        "timestamp": timestamp,
        "event_class": event_class,
        "history": list(history),
        "touch_votes": int(touch_votes),
        "class_conf_threshold": float(CLASS_CONF_THRESHOLD),
        "min_margin": float(MIN_MARGIN),
        "detections": [
            {
                "bbox": list(d["bbox"]),
                "pred_class": d["pred_class"],
                "top1": float(d["top1"]),
                "margin": float(d["margin"]),
                "probs": {k: float(v) for k, v in d["probs"].items()},
            }
            for d in detections
        ],
        "image_path": str(image_path),
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f"[SAVE] JSON path: {json_path}")

### MAIN LOOP

This section is taken from the realtime camera script and kept with comments for clarity.

In [9]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

history = deque(maxlen=ROLLING_WINDOW)
frame_count = 0
last_alert_time = 0
last_predictions = []
last_hand_result = None

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to read webcam frame.")
        break

    frame_count += 1
    display_frame = frame.copy()
    frame_h, frame_w = frame.shape[:2]

    current_vote = "NotTouching"

    # -----------------------------------------------------
    # Run detection every FRAME_SKIP frames
    # -----------------------------------------------------
    if frame_count % FRAME_SKIP == 0:
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int(time.time() * 1000)

        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)
        last_hand_result = hand_result
        last_predictions = []

        valid_touch_dets = []

        if hand_result.hand_landmarks:
            for landmarks in hand_result.hand_landmarks:
                x1, y1, x2, y2 = get_bbox_from_landmarks(landmarks, frame_w, frame_h, HAND_BOX_PADDING)
                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                pred_class, probs_dict, top1, margin = classify_crop(crop)

                det = {
                    "bbox": (x1, y1, x2, y2),
                    "pred_class": pred_class,
                    "top1": top1,
                    "margin": margin,
                    "probs": probs_dict,
                }

                if is_valid_touch_detection(det, frame_w, frame_h):
                    det["display_class"] = pred_class
                    valid_touch_dets.append(det)
                else:
                    det["display_class"] = "Unknown"

                last_predictions.append(det)

        current_vote = "Touching" if len(valid_touch_dets) > 0 else "NotTouching"
        history.append(current_vote)

    # -----------------------------------------------------
    # Draw boxes and labels
    # -----------------------------------------------------
    for item in last_predictions:
        x1, y1, x2, y2 = item["bbox"]
        display_class = item["display_class"]
        pred_class = item["pred_class"]
        top1 = item["top1"]
        margin = item["margin"]
        probs_dict = item["probs"]

        box_color = get_box_color(display_class)
        cv2.rectangle(display_frame, (x1, y1), (x2, y2), box_color, 3)

        label1 = FRIENDLY_LABELS[display_class]
        label2 = f"Top1: {top1:.2f}"
        label3 = f"Margin: {margin:.2f}"
        label4 = f"TP:{probs_dict['TouchingPlants']:.2f}  TW:{probs_dict['TouchingWildlife']:.2f}"

        y_text = max(30, y1 - 10)
        cv2.putText(display_frame, label1, (x1, y_text),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.60, box_color, 2)
        cv2.putText(display_frame, label2, (x1, y_text + 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        cv2.putText(display_frame, label3, (x1, y_text + 44),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        cv2.putText(display_frame, label4, (x1, y_text + 66),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2)

    # -----------------------------------------------------
    # Alert logic
    # -----------------------------------------------------
    counts = Counter(history)
    now = time.time()
    alert_triggered = False

    valid_touch_dets = [
        d for d in last_predictions
        if d["display_class"] in ("TouchingPlants", "TouchingWildlife")
    ]

    if counts["Touching"] >= ALERT_MIN_TOUCHING and (now - last_alert_time) > COOLDOWN_SECONDS and valid_touch_dets:
        alert_triggered = True
        last_alert_time = now

        strongest = max(valid_touch_dets, key=lambda x: x["top1"])
        event_class = strongest["pred_class"]

        alert_frame = display_frame.copy()
        draw_fullscreen_border(alert_frame, color=(0, 0, 255), thickness=18)

        print(f"[ALERT] Triggered | votes={counts['Touching']}/{ROLLING_WINDOW} | event={event_class}")
        save_alert_frame(
            frame=alert_frame,
            detections=valid_touch_dets,
            history=history,
            touch_votes=counts["Touching"],
            event_class=event_class,
            prefix="alert"
        )

    # -----------------------------------------------------
    # Global UI
    # -----------------------------------------------------
    cv2.putText(display_frame, f"Touch votes: {counts['Touching']}/{ROLLING_WINDOW}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(display_frame, f"Class conf: {CLASS_CONF_THRESHOLD:.2f}", (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(display_frame, f"Min margin: {MIN_MARGIN:.2f}", (20, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(display_frame, f"Cooldown: {COOLDOWN_SECONDS}s", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)

    if alert_triggered:
        draw_fullscreen_border(display_frame, color=(0, 0, 255), thickness=18)
        cv2.putText(display_frame, "ALERT TRIGGERED", (20, 160),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

    if last_hand_result is None or not last_predictions:
        cv2.putText(display_frame, "No hand detected", (20, frame_h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (180, 180, 180), 2)

    cv2.imshow("CTIP Realtime 2-Class Touching Monitor", display_frame)

    # -----------------------------------------------------
    # Keyboard controls
    # -----------------------------------------------------
    key = cv2.waitKey(1) & 0xFF

    if key == ord("s"):
        snapshot_frame = display_frame.copy()
        print("[MANUAL] Saving snapshot...")
        save_alert_frame(
            frame=snapshot_frame,
            detections=last_predictions,
            history=history,
            touch_votes=counts["Touching"],
            event_class="ManualSnapshot",
            prefix="manual"
        )

    if key == ord("q"):
        print("Quitting...")
        break

cap.release()
cv2.destroyAllWindows()

[MANUAL] Saving snapshot...
[SAVE] Image path: C:\COS30049 Assignment\Alerts\2026-04-17_22-12-47_manual_ManualSnapshot.jpg
[SAVE] Image save success: True
[SAVE] JSON path: C:\COS30049 Assignment\Alerts\2026-04-17_22-12-47_manual_ManualSnapshot.json
Quitting...
